<a href="https://colab.research.google.com/github/bhardwaj-amit79/Colab/blob/main/Residual_Adapter_Demo_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### <font color="blue"><b>Residual Adapters over a GPT-2 backbone (Lin et al., EMNLP Findings 2020) </b></font>

Session Objectives- By the end of this code demo, participants will be able to:

1. Understand the idea of Residual Adapter Tuning.
2. Explain the structure of a residual adapter.
3. Understand how task-specific adapters work.
4. Explain the use of segment embeddings.
5. Prepare data for multiple NLP tasks.
6. Evaluate the model using F1 and ROUGE metrics.
7. Understand text generation during prediction.
8. Explain adapter-only training with a frozen model.
9. Understand the training and validation process.
10. Interpret loss curves to evaluate learning.

In [ ]:
!pip install -q evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.3 MB/s eta 0:00:00


### <font color="blue"><b> Essential Packages/Libraries Imports </b></font>

In [ ]:
import math
import random
import re
import string
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset

### <font color="blue"><b> Function to set seeds for Reproducibility & Configuration </b></font>

In [ ]:
def set_seed(seed=42):
    random.seed(seed)                             # Python random
    np.random.seed(seed)                          # Numpy's random
    torch.manual_seed(seed)                       # torch's random
    torch.cuda.manual_seed_all(seed)              # cuda's random
    torch.backends.cudnn.deterministic = True     # Force cuDNN to use deterministic algorithms

set_seed(42)
# setting device (processing device)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAIN_SAMPLES = 400             # training samples
TEST_SAMPLES = 80               # test samples
BATCH_SIZE = 8                   # batch size or samples for one step processing
MAX_SEQ_LEN = 1024               # Hard cap for GPT-2
LEARNING_RATE = 3e-4             # learning rate
EPOCHS = 5                       # number of epochs

### <font color="blue"><b> Residual Adapter Component & Hook-Based GPT-2 Wrapper</b></font>

<img src="https://raw.githubusercontent.com/Rekha215/Demo-Images/refs/heads/main/Residual_Adapter_P.png" width="400">

<img src="https://raw.githubusercontent.com/Rekha215/Demo-Images/refs/heads/main/R_adapter_class.png" width="700">

In [ ]:
class ResidualAdapter(nn.Module):
    def __init__(self, hidden_dim: int, bottleneck_dim: int = 32):
        super().__init__()
        self.ln = nn.LayerNorm(hidden_dim)
        self.down_proj = nn.Linear(hidden_dim, bottleneck_dim)
        self.up_proj = nn.Linear(bottleneck_dim, hidden_dim)

        # near-identity init: zero the up-projection so Adapter(h) = h at start
        nn.init.normal_(self.down_proj.weight, std=0.02)
        nn.init.zeros_(self.down_proj.bias)
        nn.init.zeros_(self.up_proj.weight)
        nn.init.zeros_(self.up_proj.bias)

    def forward(self, h):
        # Adapter(H) = (ReLU(LN(H) W^E)) W^D + H
        z = F.relu(self.down_proj(self.ln(h)))
        return self.up_proj(z) + h


class GPT2BlockWithAdapter(nn.Module):
    def __init__(self, block, layer_adapters, task_holder, layer_idx): # initialize the object's attributes
        super().__init__()
        self.block = block                    # original frozen GPT2Block (registered child - fine)
        self.layer_adapters = layer_adapters  # ModuleDict: task_name -> ResidualAdapter for THIS layer
        self.layer_idx = layer_idx
        # task_holder is a plain dict (NOT an nn.Module) -> no registration, no cycle
        object.__setattr__(self, "task_holder", task_holder)

    def forward(self, *args, **kwargs):
        outputs = self.block(*args, **kwargs)
        task = self.task_holder["current"]

        if isinstance(outputs, tuple):
            hidden_states = outputs[0]
            if task is not None:
                hidden_states = self.layer_adapters[task](hidden_states)
            return (hidden_states,) + outputs[1:]

        # bare tensor - defensive fallback for the case where self.block(...) returns just a tensor instead of a tuple.
        if task is not None:
            outputs = self.layer_adapters[task](outputs)
        return outputs


class GPT2WithTaskAdapters(nn.Module):
    def __init__(self, gpt2, tasks: dict, bottleneck_dim=32): # initialize the object's attributes
        super().__init__()
        assert all(isinstance(v, int) for v in tasks.values()), \
            "tasks must map task_name -> n_segments (int)"

        self.gpt2 = gpt2
        hidden_dim = gpt2.config.n_embd
        n_layer = gpt2.config.n_layer

        # Freeze base GPT-2 weights
        for p in self.gpt2.parameters():
            p.requires_grad = False

        # Per-layer adapters: adapters_by_layer[layer_idx][task_name]
        self.adapters_by_layer = nn.ModuleList([
            nn.ModuleDict({
                name: ResidualAdapter(hidden_dim, bottleneck_dim) for name in tasks
            })
            for _ in range(n_layer)
        ])

        # Per-task segment embeddings
        self.segment_embeddings = nn.ModuleDict({
            name: nn.Embedding(n_segments, hidden_dim) for name, n_segments in tasks.items()
        })
        for emb in self.segment_embeddings.values():
            nn.init.normal_(emb.weight, std=0.02)

        # plain dict holder shared with wrappers; not an nn.Module -> no cycle
        self.task_holder = {"current": None}

        # Replace each GPT2Block with an adapter-wrapped version (in place)
        for layer_idx in range(n_layer):
            self.gpt2.transformer.h[layer_idx] = GPT2BlockWithAdapter(
                block=self.gpt2.transformer.h[layer_idx],
                layer_adapters=self.adapters_by_layer[layer_idx],
                task_holder=self.task_holder,
                layer_idx=layer_idx,
            )

    def forward(self, input_ids, segment_ids, task_name, attention_mask=None, labels=None):
        self.task_holder["current"] = task_name
        inputs_embeds = (self.gpt2.transformer.wte(input_ids) + self.segment_embeddings[task_name](segment_ids))
        outputs = self.gpt2(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels,)
        self.task_holder["current"] = None
        return outputs

    def trainable_parameters(self, task_name):
        params = []
        for layer in self.adapters_by_layer:
            params += list(layer[task_name].parameters())
        params += list(self.segment_embeddings[task_name].parameters())
        return params

### <font color="blue"><b> torch native Dataset class, samples preparation function, function to load datasets, collate function to stacking IDs and Masks per batch </b></font>

In [ ]:
class NativeTaskDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def _build_example(tokenizer, context_prefix, context_body, cue, target_str, gold_text, max_len):

    target_ids = tokenizer.encode(target_str, add_special_tokens=False) + [tokenizer.eos_token_id]

    prefix_ids = tokenizer.encode(context_prefix, add_special_tokens=False)
    cue_ids = tokenizer.encode(cue, add_special_tokens=False)
    body_ids = tokenizer.encode(context_body, add_special_tokens=False)

    # Budget for the body = max_len - target - prefix - cue - 1 (safety)
    fixed = len(target_ids) + len(prefix_ids) + len(cue_ids) + 1
    max_body_len = max(0, max_len - fixed)
    body_ids = body_ids[:max_body_len]  # truncate ONLY the body

    prompt_ids = prefix_ids + body_ids + cue_ids

    return {
        "input_ids": prompt_ids + target_ids,
        "seg_ids": [0] * len(prompt_ids) + [1] * len(target_ids),
        "labels": [-100] * len(prompt_ids) + target_ids,
        "prompt_len": len(prompt_ids),
        "gold_text": gold_text,
    }


def process_coqa(tokenizer, dataset, max_len=1024):
    processed = []
    for row in dataset:
        story = row["story"]
        questions = row["questions"]
        answers = row["answers"]["input_text"]

        for q, a in zip(questions[:1], answers[:1]):  # Evaluate first turn
            processed.append(_build_example(
                tokenizer,
                context_prefix="Context: ",
                context_body=story,
                cue=f"\nQuestion: {q}\nAnswer:",
                target_str=f" {a}",
                gold_text=a,
                max_len=max_len,
            ))
    return processed


def process_cnn_dailymail(tokenizer, dataset, max_len=1024):
    processed = []
    for row in dataset:
        article = row["article"]
        highlights = row["highlights"]

        processed.append(_build_example(
            tokenizer,
            context_prefix="Article: ",
            context_body=article,
            cue="\nTL;DR:",
            target_str=f" {highlights}",
            gold_text=highlights,
            max_len=max_len,
        ))
    return processed


def collate_fn(batch, pad_id):
    max_len = max(len(x["input_ids"]) for x in batch)

    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    seg_ids = torch.zeros((len(batch), max_len), dtype=torch.long)
    labels = torch.full((len(batch), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(batch), max_len), dtype=torch.long)

    for i, item in enumerate(batch):
        n = len(item["input_ids"])
        input_ids[i, :n] = torch.tensor(item["input_ids"])
        seg_ids[i, :n] = torch.tensor(item["seg_ids"])
        labels[i, :n] = torch.tensor(item["labels"])
        attention_mask[i, :n] = 1

    return input_ids, seg_ids, labels, attention_mask

### <font color="blue"><b> Standard Metric Calculations</b></font>

### <font color="maroon"><b>Example: Token-Level F1 Calculation (compute_f1 function)</b></font>

**Input Strings**
* **Prediction (`pred`):** `"Cats like milk."`
* **Reference (`target`):** `"Cats love milk."`

**Step 1: Text Normalization**
* **Pred Tokens (`p_toks`):** `['cats', 'like', 'milk']` *(Length = 3)*
* **Ref Tokens (`t_toks`):** `['cats', 'love', 'milk']` *(Length = 3)*

**Step 2: Token Matching (Intersection)**
* **Matching Tokens:** `['cats', 'milk']`
* **`num_same`:** **2**

**Step 3: Precision, Recall & F1 Calculation**

$$\text{Precision} = \frac{\text{num_same}}{\text{len}(p\_toks)} = \frac{2}{3}$$

$$\text{Recall} = \frac{\text{num_same}}{\text{len}(t\_toks)} = \frac{2}{3}$$

$$\text{Token F1 Score} = = \mathbf{0.6667} $$

### <font color="maroon"><b>Example: ROUGE Calculation (compute_rouge function) </b></font>

* **Prediction (`pred`):** `"Cats like milk."`
* **Reference (`target`):** `"Cats love milk."`

**Step 1: Normalizing, Stemming & Tokenization**

* **Pred Tokens:** `['cat', 'like', 'milk']`
* **Ref Tokens:** `['cat', 'love', 'milk']`

**Step 2: The 3 ROUGE Metric Calculations**

**1. ROUGE-1 (Unigrams / Single Words)**
* **Pred 1-grams:** `['cat', 'like', 'milk']`
* **Ref 1-grams:** `['cat', 'love', 'milk']`
* **Matches:** `['cat', 'milk']` *(2 matching words)*

$$\text{Precision} = \frac{2}{3}, \quad \text{Recall} = \frac{2}{3}$$

$$\text{ROUGE-1 F1} = \mathbf{66.67\%}$$


**2. ROUGE-2 (Bigrams / Consecutive Word Pairs)**
* **Pred 2-grams:** `[('cat', 'like'), ('like', 'milk')]`
* **Ref 2-grams:** `[('cat', 'love'), ('love', 'milk')]`
* **Matches:** **0 matches** *(neither pair matches because the middle word changed)*

$$\text{Precision} = \frac{0}{2}, \quad \text{Recall} = \frac{0}{2}$$

$$\text{ROUGE-2 F1} = \mathbf{0.00\%}$$


**3. ROUGE-L (Longest Common Subsequence)**
ROUGE-L finds the longest sequence of words that appear in the **same relative order**, even if other words are between them.

* **Sequence in Pred:** $\text{cat} \rightarrow \text{like} \rightarrow \text{milk}$
* **Sequence in Ref:** $\text{cat} \rightarrow \text{love} \rightarrow \text{milk}$
* **Longest Matching Subsequence (LCS):** `['cat', 'milk']` *(length 2)*

$$\text{LCS Length} = 2$$

$$\text{ROUGE-L F1} =  \mathbf{66.67\%}$$

In [ ]:
try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False

def normalize_text(s: str) -> str:
    s = s.lower() # lower the text
    s = re.sub(r'\b(a|an|the)\b', ' ', s) # remove a, an and the
    s = ''.join(ch for ch in s if ch not in set(string.punctuation)) # remove punctuations
    return ' '.join(s.split())


def compute_f1(pred: str, target: str) -> float:
    p_toks, t_toks = normalize_text(pred).split(), normalize_text(target).split()
    if not p_toks or not t_toks:
        return float(p_toks == t_toks)
    common = Counter(p_toks) & Counter(t_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(p_toks)
    recall = num_same / len(t_toks)
    return 2 * (precision * recall) / (precision + recall)


def compute_rouge(pred: str, target: str) -> dict:
    if HAS_ROUGE:
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        scores = scorer.score(target, pred)
        return {
            "ROUGE-1": scores['rouge1'].fmeasure * 100,
            "ROUGE-2": scores['rouge2'].fmeasure * 100,
            "ROUGE-L": scores['rougeL'].fmeasure * 100
        }
    else:
        f1 = compute_f1(pred, target) * 100
        return {"ROUGE-1": f1, "ROUGE-2": f1, "ROUGE-L": f1}

### <font color="blue"><b> Function to generate predictions</b></font>

In [ ]:
def generate_prediction(model, tokenizer, task_name, prompt_ids, prompt_segs, target_seg=1, max_gen_len=32):
    model.eval()

    max_p_len = 1024 - max_gen_len - 1
    prompt_ids = prompt_ids[:max_p_len]
    prompt_segs = prompt_segs[:max_p_len]

    curr_ids = torch.tensor([prompt_ids], dtype=torch.long).to(DEVICE)
    curr_segs = torch.tensor([prompt_segs], dtype=torch.long).to(DEVICE)

    generated_ids = []

    with torch.no_grad():
        for _ in range(max_gen_len):
            if curr_ids.shape[1] >= 1024:
                break

            attn_mask = torch.ones_like(curr_ids)

            # Forward call handles task setting internally
            outputs = model(curr_ids, curr_segs, task_name, attention_mask=attn_mask)

            next_token_logits = outputs.logits[0, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1).item()

            if next_token == tokenizer.eos_token_id:
                break

            generated_ids.append(next_token)
            curr_ids = torch.cat([curr_ids, torch.tensor([[next_token]], device=DEVICE)], dim=1)
            curr_segs = torch.cat([curr_segs, torch.tensor([[target_seg]], device=DEVICE)], dim=1)

    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

### <font color="blue"><b> Validation Loss + Loss Curve Plotting</b></font>

In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader, task_name):
    model.eval()
    total_loss = 0.0
    n_batches = 0
    for input_ids, seg_ids, labels, attention_mask in loader:
        input_ids, seg_ids = input_ids.to(DEVICE), seg_ids.to(DEVICE)
        labels, attention_mask = labels.to(DEVICE), attention_mask.to(DEVICE)
        outputs = model(input_ids, seg_ids, task_name, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()
        n_batches += 1
    return total_loss / max(1, n_batches)


def plot_loss_curves(history: dict, out_path="loss_curves.png"):

    tasks = list(history.keys())
    fig, axes = plt.subplots(1, len(tasks), figsize=(6 * len(tasks), 5), squeeze=False)

    for ax, task_name in zip(axes[0], tasks):
        train = history[task_name]["train"]
        val = history[task_name]["val"]
        epochs = range(1, len(train) + 1)

        ax.plot(epochs, train, marker="o", linewidth=2, label="train")
        ax.plot(epochs, val, marker="s", linewidth=2, label="val")

        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title(task_name)
        ax.set_xticks(list(range(1, EPOCHS + 1)))
        ax.grid(True, alpha=0.3)
        ax.legend()

    fig.suptitle("Residual Adapter Tuning — Train vs. Validation Loss")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)
    print(f"\nSaved loss curves -> {out_path}")

### <font color="blue"><b> Main Pipeline Execution</b></font>



<img src="https://raw.githubusercontent.com/Rekha215/Demo-Images/refs/heads/main/residual_adapters_main_pipeline.png" width="600">

In [ ]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

tasks_config = {"coqa": 2, "cnn_dailymail": 2} # each task with two segment ids
model = GPT2WithTaskAdapters(gpt2, tasks_config, bottleneck_dim=32).to(DEVICE)

coqa_train_raw = load_dataset("stanfordnlp/coqa", split=f"train[:{TRAIN_SAMPLES}]")
coqa_test_raw = load_dataset("stanfordnlp/coqa", split=f"validation[:{TEST_SAMPLES}]")

cnn_train_raw = load_dataset("abisee/cnn_dailymail", "3.0.0", split=f"train[:{TRAIN_SAMPLES}]")
cnn_test_raw = load_dataset("abisee/cnn_dailymail", "3.0.0", split=f"validation[:{TEST_SAMPLES}]")

In [ ]:
# One sample of COQA dataset
print(coqa_train_raw[0])
print("----------------------------")
print(coqa_train_raw[0]['story'])
print("----------------------------")
print(coqa_train_raw[0]['questions'])
print("----------------------------")
print(coqa_train_raw[0]['answers'])
print("----------------------------")
print(coqa_train_raw[0]['answers']['input_text'])

In [ ]:
# One sample of Daily/CNN dataset
print(cnn_train_raw[0])
print("----------------------------")
print(cnn_train_raw[0]['article'])
print("----------------------------")
print(cnn_train_raw[0]['highlights'])

In [ ]:
task_map = [
    ("coqa", process_coqa(tokenizer, coqa_train_raw), process_coqa(tokenizer, coqa_test_raw)),
    ("cnn_dailymail", process_cnn_dailymail(tokenizer, cnn_train_raw), process_cnn_dailymail(tokenizer, cnn_test_raw)),
]

loss_history = {}  # task_name -> {"train": [...], "val": [...]}

for dataset_name, train_samples, test_samples in task_map:
    print("=" * 65)
    print(f" Dataset: '{dataset_name}'")
    print("=" * 65)

    train_loader = DataLoader(
        NativeTaskDataset(train_samples),
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
    )

    # Held-out set for per-epoch validation loss (reuses the test samples).
    val_loader = DataLoader(
        NativeTaskDataset(test_samples),
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id)
    )

    optimizer = torch.optim.Adam(model.trainable_parameters(dataset_name), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )

    # Train
    train_losses, val_losses = [], []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0

        for input_ids, seg_ids, labels, attention_mask in train_loader:
            input_ids, seg_ids = input_ids.to(DEVICE), seg_ids.to(DEVICE)
            labels, attention_mask = labels.to(DEVICE), attention_mask.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(input_ids, seg_ids, dataset_name, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        val_loss = evaluate_loss(model, val_loader, dataset_name)
        train_losses.append(avg_loss)
        val_losses.append(val_loss)
        print(f"  [{dataset_name}] Epoch {epoch}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Val Loss: {val_loss:.4f}")

    loss_history[dataset_name] = {"train": train_losses, "val": val_losses}

    # Evaluate
    preds, golds = [], []
    for item in test_samples:
        p_ids = item["input_ids"][:item["prompt_len"]]
        p_segs = item["seg_ids"][:item["prompt_len"]]
        pred_text = generate_prediction(model, tokenizer, dataset_name, p_ids, p_segs)
        preds.append(pred_text)
        golds.append(item["gold_text"])

    print(f"\n  [Sample Pred]: '{preds[0]}'")
    print(f"  [Sample Gold]: '{golds[0]}'")

    if dataset_name == "coqa":
        f1_scores = [compute_f1(p, g) for p, g in zip(preds, golds)]
        avg_f1 = (sum(f1_scores) / len(f1_scores)) * 100.0
        print(f"  [{dataset_name}] TEST EVALUATION | Macro F1: {avg_f1:.2f}%\n")

    elif dataset_name == "cnn_dailymail":
        rouge_scores = [compute_rouge(p, g) for p, g in zip(preds, golds)]
        r1 = sum(s["ROUGE-1"] for s in rouge_scores) / len(rouge_scores)
        r2 = sum(s["ROUGE-2"] for s in rouge_scores) / len(rouge_scores)
        rl = sum(s["ROUGE-L"] for s in rouge_scores) / len(rouge_scores)
        print(f"  [{dataset_name}] TEST EVALUATION | ROUGE-1: {r1:.2f} | ROUGE-2: {r2:.2f} | ROUGE-L: {rl:.2f}\n")

# Plot per-task loss curves
plot_loss_curves(loss_history, out_path="loss_curves.png")

In [ ]:
# Parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

adapter_params = sum(
    p.numel()
    for layer in model.adapters_by_layer
    for adapter in layer.values()
    for p in adapter.parameters()
)

# Segment embedding parameters
segment_params = sum(
    p.numel()
    for emb in model.segment_embeddings.values()
    for p in emb.parameters()
)

adapter_segment_params = adapter_params + segment_params

print(f"Total Parameters      : {total_params:,} (100.00%)")
print(f"Trainable Parameters  : {trainable_params:,} ({100*trainable_params/total_params:.2f}%)")
print(f"Frozen Parameters     : {frozen_params:,} ({100*frozen_params/total_params:.2f}%)")
print(f"Adapter Parameters    : {adapter_params:,} ({100*adapter_params/total_params:.2f}%)")
print(f"Segment Embeddings    : {segment_params:,} ({100*segment_params/total_params:.4f}%)")
print(f"Adapter + Segment     : {adapter_segment_params:,} ({100*adapter_segment_params/total_params:.2f}%)")

### <font color="blue"><b>Future Directions</b></font>

1. Extend the framework to support additional NLP tasks beyond question answering and summarization.

2. Apply the approach to larger pre-trained language models for improved performance.

3. Compare Residual Adapter Tuning with other parameter-efficient fine-tuning methods.

4. Explore joint multi-task training to improve knowledge sharing across tasks.

5. Evaluate the framework on domain-specific datasets such as biomedical, legal, and financial text.

6. Investigate different bottleneck dimensions to optimize the trade-off between parameter efficiency and performance.

7. Evaluate various learning rates and schedulers for faster and more stable adapter training.

8. Study the effect of different projection layer configurations, including activation functions and initialization strategies.

9. Experiment with different segment embedding sizes and numbers of segment IDs for improved task representation.

10. Analyze the impact of batch size, sequence length, and number of training epochs on model performance and convergence.

In [ ]:
# Thank You!

### Adapters lib
https://adapterhub.ml/